## XGBoost
`XGBoost` (Extreme Gradient Boosting) is a powerful and efficient open-source machine learning library based on the gradient boosting framework. It is widely used for both regression and classification problems.

- Key Features:

  - Highly efficient and scalable implementation optimized for speed and performance.

  - Supports regularization (L1 and L2) to reduce overfitting.

  - Handles missing data internally.

  - Supports parallel and distributed computing.

  - Provides flexibility with various objective functions and evaluation metrics.

- Advantages:

  - Superior predictive accuracy for many machine learning tasks.

  - Effective handling of large datasets with high dimensionality.

  - Robustness against overfitting through shrinkage, tree pruning, and regularization.

Strong community support and integration with popular data science frameworks.

`XGBoost` is a go-to algorithm for many data science competitions and real-world applications due to its balance of accuracy, speed, and ease of use in predictive modeling.

In [2]:
# Import libraries for pipeline and preprocessing
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import numpy as np

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Load the dataset
df = pd.read_csv('/content/drive/MyDrive/ImpactSense_Oct25/data/earthquakes_data_preprocessed.csv')

# Prepare features and target variable
X = df.drop(columns='risk_score')
y = df['risk_score']

In [6]:
# Split into train and test sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
model = XGBRegressor(random_state=42, use_label_encoder=False, eval_metric='rmse')
model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [10:15:01] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [10]:
# Predict on validation data

y_pred = model.predict(X_val)

In [11]:
# Evaluate model performance
mse = mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

In [12]:
print(f'XGBoost Regression MSE: {mse:.4f}')
print(f'XGBoost Regression R² Score: {r2:.4f}')

XGBoost Regression MSE: 0.1403
XGBoost Regression R² Score: 0.9934


In [13]:
from sklearn.model_selection import GridSearchCV
# Define model
model = XGBRegressor(random_state=42, use_label_encoder=False, eval_metric='rmse')

# Define hyperparameter grid to search
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

# Set up GridSearchCV with 3-fold cross-validation
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    verbose=1,
    n_jobs=-1
)

# Fit on your training data
grid_search.fit(X_train, y_train)

# Get the best model and parameters after tuning
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_

print("Best Parameters: ", best_params)

Fitting 3 folds for each of 243 candidates, totalling 729 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [10:24:29] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best Parameters:  {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 50, 'subsample': 1.0}


In [14]:
# Use the best estimator to predict on test data
best_xgBoost = grid_search.best_estimator_
y_pred = best_xgBoost.predict(X_val)

In [15]:
# Evaluate on test data
from sklearn.metrics import mean_squared_error, r2_score
print("Test MSE:", mean_squared_error(y_val, y_pred))
print("Test R2:", r2_score(y_val, y_pred))

Test MSE: 0.12860669874788816
Test R2: 0.9939494638653208
